# Collections in depth and comprehensions

Collections stop being just storage containers at this level. They become part of how you express an algorithm. Lists, dictionaries, sets, and comprehensions let you shape data in compact ways, but compact code is only good when the underlying idea stays readable.

Comprehensions are especially important because they combine iteration, filtering, and transformation in one expression. Used well, they make intent obvious. Used badly, they hide logic inside a dense one-liner.

The aim of this module is to make you deliberate about both data shape and performance. When you choose a collection or a comprehension style, you are choosing both readability and computational behaviour.

## Visual model

```text
input data -> filter -> transform -> group -> result
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `list`: a dynamic array of pointers

A `list` is a contiguous array of **pointers to objects**, not the objects
themselves. That single fact explains its performance profile.

```text
lst = [a, b, c]

  lst ──> [ ● , ● , ● , _ , _ , _ ]      over-allocated
            │   │   │
            v   v   v
           obj obj obj                    scattered anywhere in memory
```


| Operation | Complexity | Note |
|---|---|---|
| `lst[i]` | O(1) | pointer arithmetic |
| `lst[i] = x` | O(1) | |
| `append` | O(1) amortized | occasional resize copies everything |
| `pop()` | O(1) | from the end |
| `pop(0)` / `insert(0, x)` | **O(n)** | everything shifts |
| `x in lst` | **O(n)** | linear scan with `==` |
| `del lst[i]` | O(n) | |
| `len` | O(1) | stored |
| `sort` | O(n log n) | Timsort, stable |
| `lst[a:b]` | O(b-a) | builds a new list |

**Amortized O(1) append** means: the list over-allocates, so most appends are
free; occasionally it grows (roughly 1.125x plus a constant in CPython) and
copies. Averaged over many appends, constant. Any *single* append may be O(n).

The pointer indirection is why a list of a million ints costs ~40 MB and a NumPy
array of the same costs 8 MB, and why numeric loops are slow (Modules 23, 29).

---

## Concept 2. `dict`: a hash table with compact ordering

Since 3.6, CPython's dict has two parts: a dense array of entries in insertion
order, and a sparse index array of positions into it.

```text
indices : [ _ , 1 , _ , 0 , _ , 2 , _ , _ ]    sparse, sized to load factor
entries : [ (hash, key, value),                dense, INSERTION ORDER
            (hash, key, value),
            (hash, key, value) ]
```


That layout gives ordering for free and saves memory. Insertion order became a
**language guarantee in 3.7** (it was a CPython implementation detail in 3.6 —
worth knowing when reading old code).

| Operation | Complexity |
|---|---|
| `d[k]`, `d[k] = v`, `del d[k]`, `k in d` | O(1) average, O(n) worst |
| iteration | O(n), insertion order |
| `len` | O(1) |

The lookup: hash the key, mask it to an index, probe. On collision, probe again.
On a match of hashes, confirm with `==`. This is why **`__hash__` and `__eq__`
must agree** (Module 09), and why a mutable key would be unfindable (Module 03).

### The methods worth knowing

```text
d.get(k, default)               # no KeyError
d.setdefault(k, [])             # get, inserting the default if absent
d.pop(k, default)
d.popitem()                     # removes and returns the LAST item (LIFO)
d | other                       # merge, 3.9+ (right wins)
d |= other                      # in-place merge
{**a, **b}                      # merge, older syntax
d.keys() / .values() / .items() # VIEWS: live, not copies
dict.fromkeys(seq)              # dedupe preserving order (Module 03)
```


Views are live and cheap:

In [ ]:
keys = d.keys()
d["new"] = 1
print("new" in keys)      # True -- the view reflects the change

Key views also support set operations: `d1.keys() & d2.keys()` gives the common
keys. `d.items() - other.items()` gives the differing pairs. Underused and
excellent.

**`setdefault` versus `defaultdict`:**

In [ ]:
groups = {}
for item in items:
    groups.setdefault(item.kind, []).append(item)     # fine

from collections import defaultdict
groups = defaultdict(list)
for item in items:
    groups[item.kind].append(item)                     # cleaner

The `defaultdict` catch: **reading a missing key inserts it.** `if x in dd`
is safe; `dd[x]` is not. Convert with `dict(dd)` before returning it to code
that does not expect that behaviour.

---

## Concept 5. Comprehensions

In [ ]:
[f(x) for x in xs if pred(x)]           # list
{f(x) for x in xs}                      # set
{k: v for k, v in pairs}                # dict
(f(x) for x in xs)                      # GENERATOR -- lazy, not a tuple

Read them outside-in: *what to produce*, then *what to loop over*, then *what to
keep*.

In [ ]:
[y for x in matrix for y in x]           # flatten: loops in the same order
                                          # you would write them nested
[[y for y in row] for row in matrix]     # nested comprehension: inner produces
                                          # a list per row

The multi-`for` order trips everyone up once. It reads left to right in the same
order as the equivalent nested `for` statements.

### Conditions

In [ ]:
[x for x in xs if x > 0]                 # FILTER: after the for
[x if x > 0 else 0 for x in xs]          # TRANSFORM: a conditional expression
                                          # before the for
[x for x in xs if x > 0 if x < 10]       # two filters, ANDed

### When not to use one

- More than two `for` clauses, or a `for` plus two conditions: use a loop.
- Any side effect. `[print(x) for x in xs]` builds a list of `None` and throws
  it away. Write a `for` loop.
- When the expression no longer fits on a line and reads worse than three lines
  of loop.

A comprehension should read as a *description of the result*. When it starts
reading as a *procedure*, it should be a loop.

### Generator expressions: the lazy version

In [ ]:
sum(x**2 for x in range(1_000_000))     # never builds the list
any(line.startswith("ERROR") for line in fh)   # stops at the first hit
max((score(x), x) for x in candidates)

Parentheses are optional when it is the only argument to a call. Use a generator
expression when you are consuming the values once — it uses O(1) memory instead
of O(n) and can short-circuit. Module 14 makes this a design tool.

---

## Concept 6. Sorting

In [ ]:
sorted(xs)                                   # new list
xs.sort()                                    # in place, returns None
sorted(xs, key=len)                          # by a computed value
sorted(xs, key=lambda p: (p.dept, -p.score)) # multi-key; - reverses a number
sorted(xs, reverse=True)
sorted(xs, key=str.casefold)                 # case-insensitive text

from operator import attrgetter, itemgetter
sorted(people, key=attrgetter("age"))        # faster and clearer than a lambda
sorted(rows, key=itemgetter(1, 0))

Facts to keep:

- **Timsort, O(n log n), and stable.** Stability means equal elements keep their
  relative order, which is what makes multi-pass sorting work:

  ```python
  rows.sort(key=itemgetter("name"))     # secondary key first
  rows.sort(key=itemgetter("dept"))     # primary key last
  ```

- The `key` function is called **once per element**, not on every comparison.
  That is why `key=` beats the removed `cmp=` and why an expensive key is fine.
- For reverse-sorting on a non-numeric key, use `reverse=True` rather than
  negating — you cannot negate a string.
- For top-k, `heapq.nlargest(k, xs)` is O(n log k) and streams its input.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `list`: a dynamic array of pointers
- Section 2: `dict`: a hash table with compact ordering
- Section 3: `set`: a hash table without values
- Section 4: `tuple`
- Section 5: Comprehensions
- Section 6: Sorting
- Section 7: `collections`
- Section 8: Choosing a container

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import timeit
from collections import deque

PREDICTIONS = """
Operation                          | Your prediction (O(?)) | Measured growth
-----------------------------------|------------------------|----------------
list[i]                            |                        |
list.append                        |                        |
list.insert(0, x)                  |                        |
list.pop()                         |                        |
list.pop(0)                        |                        |
x in list                          |                        |
x in set                           |                        |
x in dict                          |                        |
dict[k] = v                        |                        |
deque.appendleft                   |                        |
sorted(list)                       |                        |
list[:] (full slice copy)          |                        |
"""


# TODO 1 -----------------------------------------------------------------------

---

## `measure`

Time `stmt` at each size and return per-operation times in microseconds.

In [ ]:
def measure(setup: str, stmt: str, sizes: list[int]) -> list[float]:
    """Time `stmt` at each size and return per-operation times in microseconds.

    Use timeit.timeit with a `number` chosen so each measurement takes roughly
    a tenth of a second -- too few repetitions and the timer noise dominates,
    too many and you wait forever.

    Hint: timeit.timeit(stmt, setup, number=N) returns TOTAL seconds for N
    repetitions.
    """
    raise NotImplementedError

---

## `growth_ratio`

Given timings at sizes n, 2n, 4n, 8n..., return the ratio between

In [ ]:
def growth_ratio(times: list[float]) -> list[float]:
    """Given timings at sizes n, 2n, 4n, 8n..., return the ratio between
    consecutive measurements.

    Interpreting the ratios is the whole exercise:
      ~1.0  -> O(1)        the size did not matter
      ~2.0  -> O(n)        doubling the input doubled the time
      ~2.2  -> O(n log n)  slightly worse than doubling
      ~4.0  -> O(n^2)      doubling the input quadrupled the time
    """
    raise NotImplementedError

---

## `run_all`

Measure all twelve operations across at least four sizes and print a

In [ ]:
def run_all() -> None:
    """Measure all twelve operations across at least four sizes and print a
    table of times and growth ratios.

    Watch out for these measurement traps -- each one will give you a wrong
    answer if you miss it:

    a) DESTRUCTIVE operations. Timing `lst.pop()` a million times empties the
       list, and then you are timing IndexError handling. Rebuild the structure
       in the setup, or measure a fixed number of pops against a fresh copy.

    b) The SETUP is not timed, so building the list in setup is free. Make sure
       the work you want to measure is in stmt and nothing else is.

    c) MEMBERSHIP tests need a MISSING element to measure the worst case. `0 in
       lst` finds it immediately and measures nothing. Search for something that
       is not there.

    d) SORTING an already-sorted list is O(n) with Timsort, not O(n log n) --
       Timsort detects existing runs. Shuffle first, and use a fresh shuffled
       copy per repetition.
    """
    raise NotImplementedError

---

## `amortization_demo`

Show that append is amortized O(1) by finding the resize events.

In [ ]:
def amortization_demo() -> None:
    """Show that append is amortized O(1) by finding the resize events.

    Append to a list one element at a time, recording sys.getsizeof(lst) after
    each append. Print only the sizes at which the allocated size CHANGED.

    Then answer:
      - what is the pattern of the growth?  (compute each jump as a ratio)
      - how many resizes happen in a million appends? Roughly.
      - therefore, what fraction of appends pay the copy cost?
      - and so: why is "amortized O(1)" honest rather than a fudge?
    """
    raise NotImplementedError

---

## `crossover_point`

Find where `set` beats `list` for membership.

In [ ]:
def crossover_point() -> None:
    """Find where `set` beats `list` for membership.

    For n = 1, 2, 4, 8, ... 4096, time k membership tests against a list of n
    items and against a set of n items, INCLUDING the cost of building the set.

    Report the n at which the set version becomes faster.

    The answer is smaller than most people guess, and it is the practical
    justification for "just use a set".
    """
    raise NotImplementedError

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    print(PREDICTIONS)
    run_all()
    amortization_demo()
    crossover_point()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.